# IMPORT LIBRARIES AND LOAD DATA

In [1]:
#Import libraries and load data
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from category_encoders.target_encoder import TargetEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler


transactions = pd.read_csv('trans_labelled_clean.csv')
cards = pd.read_csv('cards_data_south_africa.csv')
users = pd.read_csv('user_data_south_africa.csv')

In [2]:
trans_merged = pd.merge(transactions, cards, left_on='card_id', right_on='id')
trans_merged = pd.merge(trans_merged, users, left_on='client_id_x', right_on='id')
trans_merged = trans_merged.drop(columns=['id_y', 'client_id_y', 'id'])
trans_merged = trans_merged.rename(columns={
    'id_x': 'id',
    'client_id_x': 'user_id',
})
trans_merged['is_negative_amount'] = trans_merged['amount'] < 0
trans_merged['abs_amount'] = trans_merged['amount'].abs()
trans_merged['is_negative_amount'] = trans_merged['is_negative_amount'].astype(int)
trans_merged['is_fraud'] = trans_merged['is_fraud'].astype(int)

In [3]:
#Absolute amount
trans_merged['is_negative_amount'] = trans_merged['amount'] < 0
trans_merged['abs_amount'] = trans_merged['amount'].abs()

In [4]:
trans_merged['is_negative_amount'] = trans_merged['is_negative_amount'].astype(int)

In [5]:
trans_merged.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'current_age',
       'retirement_age', 'birth_year', 'birth_month', 'gender', 'address',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'is_negative_amount', 'abs_amount'],
      dtype='object')>

# Transaction Behavioral Features

In [7]:
from datetime import datetime, timedelta


trans_merged["date"] = pd.to_datetime(trans_merged["date"])

df = trans_merged
# From 'date'
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
df['month'] = df['date'].dt.month

# Time since last transaction per card/user
df = df.sort_values(['card_id', 'date'])
df['time_since_last_tx'] = df.groupby('card_id')['date'].diff().dt.total_seconds() / 60  # minutes

In [8]:
# Already have: 'amount', 'abs_amount', 'is_negative_amount'
df['amount_to_limit_ratio'] = df['abs_amount'] / df['credit_limit']
df['amount_to_income_ratio'] = df['abs_amount'] / df['yearly_income'].replace(0, 1)
df['is_large_tx'] = (df['abs_amount'] > df['credit_limit'] * 0.5).astype(int)  # >50% of limit
df['is_micro_tx'] = (df['abs_amount'] < 1).astype(int)  # Testing/skimming

In [9]:
df['use_chip'].value_counts()

use_chip
swipe transaction     4617262
chip transaction      2990316
online transaction    1007955
Name: count, dtype: int64

In [12]:
df["acct_open_date"] = pd.to_datetime(df["acct_open_date"])

C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_6880\2628324972.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["acct_open_date"] = pd.to_datetime(df["acct_open_date"])


In [13]:
# Card age
df['card_age_days'] = (df['date'] - df['acct_open_date']).dt.days

# Chip usage anomaly
df['chip_mismatch'] = ((df['has_chip'] == True) & (df['use_chip'] == 'swipe_transaction')).astype(int)
df['chip_used'] = (df['use_chip'] == 'chip transaction').astype(int)

# Time since PIN change
df['years_since_pin_change'] = df['date'].dt.year - df['year_pin_last_changed']
df['pin_changed_recently'] = (df['years_since_pin_change'] <= 1).astype(int)

In [ ]:
'''rans_merged['date'] = pd.to_datetime(trans_merged['date'])
trans_merged = trans_merged.sort_values(['user_id', 'date'])

# Set date as index
trans_merged = trans_merged.set_index('date')

trans_merged['user_txn_count_1h'] = (
    trans_merged
        .groupby('user_id')
        .rolling('1h')['amount']   # now index is DatetimeIndex → '1h' is valid
        .count()
        .reset_index(level=0, drop=True)
)

trans_merged['user_txn_count_24h'] = (
    trans_merged
        .groupby('user_id')
        .rolling('24h')['amount']
        .count()
        .reset_index(level=0, drop=True)
)

# If you need 'date' as a column again:
trans_merged = trans_merged.reset_index()'''

# Customer Profile & Velocity Features

In [17]:
# Window functions - need to sort by date first
df = df.sort_values(['user_id', 'date'])

# Transactions in last 24 hours
df['tx_count_24h'] = df.groupby('user_id').rolling('24h', on='date')['id'].count().values

# Amount spent in last 24 hours
df['amount_24h'] = df.groupby('user_id').rolling('24h', on='date')['abs_amount'].sum().values

# Unique merchants in last 24 hours
# df['unique_merchants_24h'] = df.groupby('user_id').rolling('24h', on='date')['merchant_id'].nunique().values

# Same for 7 days
df['tx_count_7d'] = df.groupby('user_id').rolling('7d', on='date')['id'].count().values

In [18]:
# User's typical transaction amount
user_avg_amount = df.groupby('user_id')['abs_amount'].transform('mean')
user_std_amount = df.groupby('user_id')['abs_amount'].transform('std')
df['amount_z_score'] = (df['abs_amount'] - user_avg_amount) / (user_std_amount + 1e-6)

# User's typical time of day
user_hour_avg = df.groupby('user_id')['hour'].transform('mean')
df['hour_deviation'] = abs(df['hour'] - user_hour_avg)

In [19]:
df['debt_to_income'] = df['total_debt'] / (df['yearly_income'] + 1)
df['credit_utilization'] = df['total_debt'] / (df['credit_limit'] * df['num_credit_cards'] + 1)
df['income_vs_local'] = df['yearly_income'] / (df['per_capita_income'] + 1)

# Age features
df['years_to_retirement'] = df['retirement_age'] - df['current_age']
df['is_young_account'] = (df['current_age'] < 25).astype(int)

# Risk Flags & External Signals

In [21]:
# Check what values exist in the problematic columns first
print(df['card_on_dark_web'].unique())
print(df['num_cards_issued'].dtype)
print(df['acct_open_date'].dtype)

# Fix 1: Compromised card indicator - handle string values
df['dark_web_risk'] = df['card_on_dark_web'].map({'Yes': 1, 'True': 1, 'Y': 1, 'yes': 1, 'true': 1, True: 1}).fillna(0).astype(int)

# Or more robust version:
def convert_to_bool(val):
    if pd.isna(val):
        return 0
    if isinstance(val, (bool, np.bool_)):
        return int(val)
    if isinstance(val, (int, float, np.number)):
        return int(bool(val))
    if isinstance(val, str):
        return int(val.lower() in ['yes', 'true', 'y', '1'])
    return 0

df['dark_web_risk'] = df['card_on_dark_web'].apply(convert_to_bool)

# Fix 2: Multiple cards issued - ensure it's numeric first
df['num_cards_issued'] = pd.to_numeric(df['num_cards_issued'], errors='coerce').fillna(0)
df['many_cards_flag'] = (df['num_cards_issued'] > 3).astype(int)

# Fix 3: Recent account opening - ensure dates are datetime
df['acct_open_date'] = pd.to_datetime(df['acct_open_date'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Now calculate account age
df['account_age_days'] = (df['date'] - df['acct_open_date']).dt.days
df['account_age_days'] = df['account_age_days'].fillna(0)  # Handle missing values

# Handle negative values (if any)
df['account_age_days'] = df['account_age_days'].clip(lower=0)

df['is_new_account'] = (df['account_age_days'] < 30).astype(int)

['No']
int64
datetime64[ns]


In [25]:
df['description'].unique()

array(['Eating Places and Restaurants',
       'Lighting, Fixtures, Electrical Supplies',
       'Drinking Places (Alcoholic Beverages)',
       'Drug Stores and Pharmacies', 'Fast Food Restaurants',
       'Discount Stores', 'Department Stores', 'Service Stations',
       'Automotive Service Shops', 'Postal Services - Government Only',
       'Travel Agencies', 'Miscellaneous Food Stores',
       'Grocery Stores, Supermarkets',
       'Semiconductors and Related Devices',
       'Lumber and Building Materials',
       'Utilities - Electric, Gas, Water, Sanitary', 'Book Stores',
       'Money Transfer', 'Miscellaneous Home Furnishing Stores',
       'Telecommunication Services', 'Artist Supply Stores, Craft Shops',
       'Wholesale Clubs', 'Taxicabs and Limousines', 'Ironwork',
       'Cable, Satellite, and Other Pay Television Services',
       'Non-Precious Metal Services',
       'Motor Freight Carriers and Trucking',
       'Amusement Parks, Carnivals, Circuses', 'Motion Picture T

In [26]:
# Define high-risk merchant categories based on fraud patterns
high_risk_categories = {
    'Money Transfer',
    'Betting (including Lottery Tickets, Casinos)',
    'Digital Goods - Games',
    'Digital Goods - Media, Books, Apps',
    'Travel Agencies',
    'Cable, Satellite, and Other Pay Television Services',
    'Precious Stones and Metals',
    'Telecommunication Services',
    'Gambling',  # if exists
    'Casino',  # if exists
}

# Create high-risk flag
df['is_high_risk_merchant'] = df['description'].isin(high_risk_categories).astype(int)

# Create risk tiers
risk_tier_mapping = {
    'Money Transfer': 3,
    'Betting (including Lottery Tickets, Casinos)': 3,
    'Digital Goods - Games': 3,
    'Digital Goods - Media, Books, Apps': 2,
    'Travel Agencies': 2,
    'Cable, Satellite, and Other Pay Television Services': 2,
    'Precious Stones and Metals': 2,
    'Telecommunication Services': 1,
    'Non-Ferrous Metal Foundries': 1,
    'Cruise Lines': 2,
    'Airlines': 2,
    # Default for others is 0
}

df['merchant_risk_tier'] = df['description'].map(risk_tier_mapping).fillna(0)

In [27]:
# Group merchants into broader categories
def categorize_merchant(desc):
    desc_lower = str(desc).lower()
    
    if any(word in desc_lower for word in ['restaurant', 'eating', 'drinking', 'fast food']):
        return 'food_beverage'
    elif any(word in desc_lower for word in ['store', 'shop', 'retail']):
        return 'retail'
    elif any(word in desc_lower for word in ['digital', 'online', 'internet', 'cable', 'tele']):
        return 'digital_services'
    elif any(word in desc_lower for word in ['travel', 'airline', 'cruise', 'hotel', 'motel']):
        return 'travel'
    elif any(word in desc_lower for word in ['medical', 'hospital', 'doctor', 'dentist']):
        return 'medical'
    elif any(word in desc_lower for word in ['auto', 'car', 'vehicle']):
        return 'automotive'
    elif any(word in desc_lower for word in ['financial', 'bank', 'insurance', 'money']):
        return 'financial'
    elif any(word in desc_lower for word in ['gambling', 'betting', 'casino', 'lottery']):
        return 'gambling'
    else:
        return 'other'

df['merchant_category_group'] = df['description'].apply(categorize_merchant)

In [28]:
# Merchant visit frequency per user
df['user_merchant_freq'] = df.groupby(['user_id', 'description']).cumcount() + 1

# First time at this merchant for user
df['is_new_merchant_for_user'] = (df['user_merchant_freq'] == 1).astype(int)

# Time since last visit to this merchant
df = df.sort_values(['user_id', 'description', 'date'])
df['time_since_last_merchant_visit'] = df.groupby(['user_id', 'description'])['date'].diff().dt.total_seconds() / 3600  # hours

In [33]:
# Create binary flags for each error type
error_types = {
    'bad_card_number': 'bad card number',
    'insufficient_balance': 'insufficient balance',
    'technical_glitch': 'technical glitch',
    'bad_zipcode': 'bad zipcode',
    'bad_cvv': 'bad cvv',
    'bad_pin': 'bad pin',
    'bad_expiration': 'bad expiration'
}

for feature_name, error_string in error_types.items():
    df[f'error_{feature_name}'] = df['errors'].str.contains(error_string, na=False).astype(int)

# Create a clean error flag
df['has_errors'] = (df['errors'] != 'No Errors').astype(int)

In [34]:
# Track error history per card/user
df = df.sort_values(['card_id', 'date'])

# Flag for consecutive error transactions
df['consecutive_errors'] = (
    df.groupby('card_id')['has_errors']
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

# Flag for error after successful transaction
df['error_after_success'] = 0
for card_id in df['card_id'].unique():
    card_mask = df['card_id'] == card_id
    error_series = df.loc[card_mask, 'has_errors']
    # Find positions where error follows success
    error_after_success = (error_series == 1) & (error_series.shift(1) == 0)
    df.loc[card_mask, 'error_after_success'] = error_after_success.astype(int)

In [35]:
df.head(20)

,id,date,user_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,error_bad_card_number,error_insufficient_balance,error_technical_glitch,error_bad_zipcode,error_bad_cvv,error_bad_pin,error_bad_expiration,has_errors,consecutive_errors,error_after_success
2408916,11835340,2022-01-01 23:00:54,1362,0,358.20,swipe transaction,82981,cape town,western cape,8000.0,...,0,0,0,0,0,0,0,0,0.0,0
7317998,20815948,2022-01-02 01:18:59,1362,0,4270.86,chip transaction,94257,east london,eastern cape,5200.0,...,0,0,0,0,0,0,0,0,0.0,0
8569644,23124672,2022-01-02 21:09:28,1362,0,-1584.00,chip transaction,43293,durban,kwazulu-natal,4000.0,...,0,0,0,0,0,0,0,0,0.0,0
5491698,17462772,2022-01-02 22:35:11,1362,0,296.64,swipe transaction,88269,pretoria,gauteng,1.0,...,0,0,0,0,0,0,0,0,0.0,0
5082614,16714756,2022-01-03 03:44:42,1362,0,33.12,chip transaction,10062,nelspruit,mpumalanga,1200.0,...,0,0,0,0,0,0,0,0,0.0,0
6503706,19318189,2022-01-03 15:55:08,1362,0,2854.62,swipe transaction,52923,port elizabeth,eastern cape,6000.0,...,0,0,0,0,0,0,0,0,0.0,0
738425,8798249,2022-01-04 05:48:32,1362,0,-1314.00,swipe transaction,61195,johannesburg,gauteng,2000.0,...,0,0,0,0,0,0,0,0,0.0,0
5706432,17856311,2022-01-04 08:10:12,1362,0,293.58,chip transaction,61195,rustenburg,north west,300.0,...,0,0,0,0,0,0,0,0,0.0,0
1415698,10029266,2022-01-04 09:30:36,1362,0,-5256.00,swipe transaction,15574,pretoria,gauteng,1.0,...,0,0,0,0,0,0,0,0,0.0,0
2071957,11223943,2022-01-04 10:07:21,1362,0,307.80,swipe transaction,10792,nelspruit,mpumalanga,1200.0,...,0,0,0,0,0,0,0,0,0.0,0


In [37]:
# View only fraud transactions
fraud_df = df[df['is_fraud'] == 1]
print(f"Total fraud transactions: {len(fraud_df)}")
print(f"Fraud rate: {len(fraud_df)/len(df):.2%}")
print("\nFraud transactions:")
fraud_df.head(50)

Total fraud transactions: 12816
Fraud rate: 0.15%

Fraud transactions:


,id,date,user_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,error_bad_card_number,error_insufficient_balance,error_technical_glitch,error_bad_zipcode,error_bad_cvv,error_bad_pin,error_bad_expiration,has_errors,consecutive_errors,error_after_success
4857751,16304149,2022-08-22 03:17:36,556,2,923.58,chip transaction,11593,johannesburg,gauteng,2000.0,...,0,0,0,0,0,0,0,0,0.0,0
5640491,17734933,2022-10-23 06:26:17,556,2,5610.60,online transaction,86616,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
4857407,16303507,2024-03-25 06:13:05,556,2,681.48,online transaction,18563,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5641170,17736183,2024-06-25 13:09:02,556,2,2390.04,online transaction,51397,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5909178,18228012,2022-03-22 03:19:51,1981,4,203.76,online transaction,60569,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5909347,18228313,2024-05-07 03:51:15,1981,4,2775.24,online transaction,34702,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5909047,18227771,2024-12-13 13:10:44,1981,4,1428.84,online transaction,27092,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5526525,17526397,2023-05-13 01:55:42,619,5,28.08,online transaction,27092,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5526208,17525792,2023-12-10 20:25:57,619,5,36.72,online transaction,76639,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0
5641051,17735964,2022-03-02 17:08:49,1783,10,1846.44,online transaction,27092,online,ONLINE,ONLINE,...,0,0,0,0,0,0,0,0,0.0,0


In [38]:
df.columns

Index(['id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'current_age',
       'retirement_age', 'birth_year', 'birth_month', 'gender', 'address',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'is_negative_amount', 'abs_amount', 'hour',
       'day_of_week', 'is_weekend', 'is_night', 'month', 'time_since_last_tx',
       'amount_to_limit_ratio', 'amount_to_income_ratio', 'is_large_tx',
       'is_micro_tx', 'card_age_days', 'chip_mismatch', 'chip_used',
       'years_since_pin_change', 'pin_changed_recently', 'tx_count_24h',
       'amount_24h', 'tx_count_7d', 'amount_z_score', 'hour_deviation',
       'debt_to_income', 'credit_

In [42]:
def add_features_working(df):
    """Working version that avoids complex rolling issues"""
    
    print("🔄 Adding features with simplified approach...")
    
    # Create a copy and ensure date is datetime
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    # Create a unique transaction timestamp for sorting
    df['timestamp'] = df['date'].astype('int64') // 10**9  # Convert to seconds
    
    # 1. SIMPLE HOUR-BASED VELOCITY (No rolling windows)
    print("Step 1: Calculating hour-based velocity...")
    
    # Group by card and hour
    df['date_hour'] = df['date'].dt.floor('H')
    df['card_tx_count_this_hour'] = df.groupby(['card_id', 'date_hour'])['id'].transform('count')
    df['card_amount_this_hour'] = df.groupby(['card_id', 'date_hour'])['abs_amount'].transform('sum')
    
    # User velocity
    df['user_tx_count_this_hour'] = df.groupby(['user_id', 'date_hour'])['id'].transform('count')
    
    # 2. CARD TESTING ALERTS
    print("Step 2: Creating card testing alerts...")
    df['card_testing_alert'] = (df['card_tx_count_this_hour'] > 3).astype(int)
    
    # Calculate time since last transaction
    df = df.sort_values(['card_id', 'date'])
    df['time_since_last_tx'] = df.groupby('card_id')['date'].diff().dt.total_seconds() / 60
    df['time_since_last_tx'] = df['time_since_last_tx'].fillna(999)
    
    df['rapid_fire_alert'] = (
        (df['card_tx_count_this_hour'] > 2) & 
        (df['time_since_last_tx'] < 2)
    ).astype(int)
    
    # 3. SIMPLE AVERAGE (last 5 transactions)
    print("Step 3: Calculating average amounts...")
    df = df.sort_values(['user_id', 'date'])
    df['last_5_amounts'] = df.groupby('user_id')['abs_amount'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    df['amount_vs_last_5_avg'] = df['abs_amount'] / df['last_5_amounts'].replace(0, 1)
    
    # 4. GEOGRAPHIC FEATURES
    print("Step 4: Adding geographic features...")
    df['prev_state'] = df.groupby('user_id')['merchant_state'].shift(1)
    df['new_state_flag'] = (
        (df['merchant_state'] != df['prev_state']) & 
        (df['prev_state'].notna())
    ).astype(int)
    
    # Count state changes in last 24 hours (simplified)
    df['date_day'] = df['date'].dt.date
    df['state_changes_today'] = df.groupby(['user_id', 'date_day'])['new_state_flag'].transform('cumsum')
    
    # 5. TIME-BASED RISK FLAGS
    print("Step 5: Adding time-based risk flags...")
    df['hour'] = df['date'].dt.hour
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Create high-risk merchant flag if not exists
    if 'is_high_risk_merchant' not in df.columns:
        high_risk_categories = ['Money Transfer', 'Betting', 'Digital Goods', 'Casino']
        df['is_high_risk_merchant'] = df['description'].str.contains(
            '|'.join(high_risk_categories), case=False
        ).astype(int)
    
    df['high_risk_time_combo'] = (
        (df['is_night'] == 1) & 
        (df['is_high_risk_merchant'] == 1)
    ).astype(int)
    
    df['is_large_tx'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)
    df['weekend_high_amount'] = (
        (df['is_weekend'] == 1) & 
        (df['is_large_tx'] == 1)
    ).astype(int)
    
    # 6. ERROR FEATURES
    if 'has_errors' in df.columns:
        print("Step 6: Adding error features...")
        df['errors_last_hour'] = df.groupby(['card_id', 'date_hour'])['has_errors'].transform('sum')
        df['high_error_velocity'] = (df['errors_last_hour'] > 2).astype(int)
    
    # Clean up temporary columns
    df = df.drop(columns=['timestamp', 'date_hour', 'date_day'])
    
    print(f"✅ Successfully added features")
    print(f"📊 New features: {[col for col in df.columns if 'alert' in col or 'flag' in col or 'velocity' in col]}")
    
    return df

In [43]:
# Run this to complete your feature set
print("🔄 Adding critical missing features...")
df = add_features_working(df)

# Check what you have now
print(f"\n✅ Total features: {len(df.columns)}")
print(f"📊 Sample of new features:")
print(df[['card_tx_count_1h', 'card_testing_alert', 'new_state_flag']].head())

🔄 Adding critical missing features...
🔄 Adding features with simplified approach...
Step 1: Calculating hour-based velocity...


C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_6880\3964891915.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['date_hour'] = df['date'].dt.floor('H')


Step 2: Creating card testing alerts...


MemoryError: Unable to allocate 3.27 GiB for an array with shape (51, 8615533) and data type int64

In [ ]:
# Save the complete feature set
df.to_csv('complete_fraud_features.csv', index=False)
print("💾 Saved complete feature set to 'complete_fraud_features.csv'")

# Ouliers and Z-Scores

In [ ]:
trans_merged['date'] = pd.to_datetime(trans_merged['date'])
trans_merged = trans_merged.sort_values('date').reset_index(drop=True)

def iqr_outlier_mask(s: pd.Series, k: float = 1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    return (s < lower_bound) | (s > upper_bound), lower_bound, upper_bound

mask_iqr, lower_iqr, upper_iqr = iqr_outlier_mask(trans_merged['abs_amount'], k=1.5)
df_iqr_outliers = trans_merged.loc[mask_iqr, ["abs_amount"]]
df_iqr_outliers.head(40).sort_values(by='abs_amount')

,abs_amount
785,2880.00
852,3002.40
450,3054.78
944,3133.62
354,3192.84
699,3234.96
832,3311.64
224,3600.00
180,3739.32
278,3768.48


In [ ]:
def zscore_outlier_mask(s: pd.Series, threshold: float = 3.5):
    z = (s - s.mean()) / s.std()
    mask = z.abs() > threshold
    return mask, z

mask_z, z = zscore_outlier_mask(trans_merged['abs_amount'], threshold=3.0)
df_z_outliers = trans_merged.loc[mask_z, ["abs_amount"]]
df_z_outliers.head(30).sort_values(by='abs_amount')

,abs_amount
1374,5038.56
714,5153.58
1811,5497.74
971,5598.00
1803,5724.00
229,5778.00
857,5958.00
1256,5970.78
646,6096.60
331,6156.00


In [ ]:
trans_merged['is_high_amount'] = mask_iqr.astype(int)

In [ ]:
trans_merged['is_zscore_outlier'] = mask_z.astype(int)

In [ ]:
trans_merged['monthly_income'] = trans_merged['yearly_income'] / 12
trans_merged["amt_to_income_ratio"] = (trans_merged["abs_amount"] / trans_merged["yearly_income"]) * 100

# Time Window Features

In [ ]:
def time_window_features(trans_merged, key, amount_col='abs_amount', ts_col='date'):
    trans_merged = trans_merged.sort_values([key, ts_col])
    g = trans_merged.groupby(key, group_keys=False)

    # Past 24 hours
    roll_24h_count = (g.rolling('24h', on=ts_col)[amount_col].count().shift(1).values)
    roll_24h_sum   = (g.rolling('24h', on=ts_col)[amount_col].sum().shift(1).values)

    # Past 7 days
    roll_7d_count = (g.rolling('7d', on=ts_col)[amount_col].count().shift(1).values)
    roll_7d_sum   = (g.rolling('7d', on=ts_col)[amount_col].sum().shift(1).values)

    return (roll_24h_count, roll_24h_sum, roll_7d_count, roll_7d_sum)

(
    trans_merged['user_txn_count_24h'],
    trans_merged['user_amt_sum_24h'],
    trans_merged['user_txn_count_7d'],
    trans_merged['user_amt_sum_7d'],
) = time_window_features(trans_merged, key='user_id')
(
    trans_merged['merchant_txn_count_24h'],
    trans_merged['merchant_amt_sum_24h'],
    trans_merged['merchant_txn_count_7d'],
    trans_merged['merchant_amt_sum_7d'],
) = time_window_features(trans_merged, key='merchant_id')

(
    trans_merged['city_txn_count_24h'],
    trans_merged['city_amt_sum_24h'],
    trans_merged['city_txn_count_7d'],
    trans_merged['city_amt_sum_7d'],
) = time_window_features(trans_merged, key='merchant_city')

In [ ]:
trans_merged['user_mcc_seen_before'] = (
    trans_merged.sort_values(['user_id', 'date'])
      .groupby(['user_id', 'mcc'])
      .cumcount()
)

trans_merged['is_first_time_user_mcc'] = (trans_merged['user_mcc_seen_before'] == 0).astype(int)

In [ ]:
#by merchant
trans_merged['user_merchant_seen_before'] = trans_merged.groupby(['user_id','merchant_id']).cumcount()
trans_merged['is_first_time_user_merchant'] = (trans_merged['user_merchant_seen_before'] == 0).astype(int)

#by city
trans_merged['user_city_seen_before'] = trans_merged.groupby(['user_id','merchant_city']).cumcount()
trans_merged['is_first_time_user_city'] = (trans_merged['user_city_seen_before'] == 0).astype(int)

#by hour
trans_merged['user_hour_seen_before'] = trans_merged.groupby(['user_id','trans_hour']).cumcount()
trans_merged['is_first_time_user_hour'] = (trans_merged['user_hour_seen_before'] == 0).astype(int)

# Geographical features

In [ ]:
trans_merged['user_city'] = trans_merged['address'].str.split(',').str[1].str.strip()

In [ ]:
city_to_region = {
    "cape town": "Western Cape",
    "pretoria": "Gauteng",
    "johannesburg": "Gauteng",
    "durban": "KwaZulu-Natal",
    "pietermaritzburg": "KwaZulu-Natal",
    "gqeberha": "Eastern Cape",
    "east london": "Eastern Cape",
    "bloemfontein": "Free State",
}

# Normalization
def normalize_city(s):
    if pd.isna(s): return None
    return (str(s).strip().lower()
            .replace('.', '')
            .replace('-', ' ')
            .replace(',', ''))

# Apply
trans_merged['city_norm'] = trans_merged['user_city'].map(normalize_city)
trans_merged['user_region'] = trans_merged['city_norm'].map(city_to_region).fillna("unknown")

In [ ]:
trans_merged['user_region'] = trans_merged['user_region'].map(normalize_city)

In [ ]:
trans_merged['is_far_from_home'] = (
    (trans_merged['use_chip'] != 'online transaction') &
    (trans_merged['user_region'] != trans_merged['merchant_state'])
).astype(int)

In [ ]:
trans_merged['is_retired'] = (trans_merged['current_age'] > trans_merged['retirement_age']).astype(int)

# Missing features

In [ ]:
trans_merged = trans_merged.sort_values("date")

user_stats = (
    trans_merged.groupby("user_id")["abs_amount"]
      .expanding()
      .agg(["mean", "std"])
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["z_amount_user"] = (
    (trans_merged["abs_amount"] - user_stats["mean"]) /
    (user_stats["std"].fillna(0) + 1e-6)
)


In [ ]:
user_median = (
    trans_merged.groupby("user_id")["abs_amount"]
      .expanding()
      .median()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["amount_user_median_ratio"] = trans_merged["abs_amount"] / (user_median + 1e-6)


In [ ]:
user_hour_mode = (
    trans_merged.groupby("user_id")["trans_hour"]
      .expanding()
      .apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["hour_deviation_user"] = abs(trans_merged["trans_hour"] - user_hour_mode)

In [ ]:
#trans_merged = trans_merged.set_index("date")

#trans_merged["user_txn_10m"] = (
#    trans_merged.groupby("user_id")
#      .rolling("10min")
#      .count()
#      .shift(1)
#      .reset_index(level=0, drop=True)
#)

In [ ]:
# Ensure sorted and datetime index
'''trans_merged = trans_merged.sort_values("date")

unique_merchants_1h = (
    trans_merged
    .groupby("user_id", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("date")["merchant_id"]
             .rolling("1h")
             .nunique()
             .shift(1)
        )
    )
)

trans_merged["unique_merchants_1h"] = unique_merchants_1h.values
trans_merged["unique_merchants_1h"] = trans_merged["unique_merchants_1h"].fillna(0)'''

'trans_merged = trans_merged.sort_values("date")\n\nunique_merchants_1h = (\n    trans_merged\n    .groupby("user_id", group_keys=False)\n    .apply(\n        lambda g: (\n            g.set_index("date")["merchant_id"]\n             .rolling("1h")\n             .nunique()\n             .shift(1)\n        )\n    )\n)\n\ntrans_merged["unique_merchants_1h"] = unique_merchants_1h.values\ntrans_merged["unique_merchants_1h"] = trans_merged["unique_merchants_1h"].fillna(0)'

In [ ]:
trans_merged.head()

,id,date,user_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,user_hour_seen_before,is_first_time_user_hour,user_city,city_norm,user_region,is_far_from_home,is_retired,z_amount_user,amount_user_median_ratio,hour_deviation_user
0,15626211,2022-01-01 00:00:10,24,2827,47.16,chip transaction,14528,polokwane,limpopo,700.0,...,0,1,Cape Town,cape town,western cape,1,0,-0.621845,0.103516,5.0
1,17529598,2022-01-01 00:00:21,1326,5561,123.12,chip transaction,94123,east london,eastern cape,5200.0,...,0,1,Pretoria,pretoria,gauteng,1,0,-1.110946,0.091505,3.0
2,15750655,2022-01-01 00:00:21,696,4207,25.56,swipe transaction,46284,bloemfontein,free state,9300.0,...,0,1,East London,east london,eastern cape,1,1,-0.590319,0.061472,17.0
3,18999635,2022-01-01 00:00:29,1640,4967,527.40,swipe transaction,81536,port elizabeth,eastern cape,6000.0,...,0,1,Pretoria,pretoria,gauteng,1,0,-0.629265,0.379878,1.0
4,18403665,2022-01-01 00:01:10,185,4046,55.44,chip transaction,22204,kimberley,northern cape,8300.0,...,0,1,Bloemfontein,bloemfontein,free state,1,0,-0.705288,0.122222,17.0


In [ ]:
trans_merged = trans_merged.reset_index(drop=False)
trans_merged["time_diff"] = trans_merged.groupby("user_id")["date"].diff().dt.seconds

trans_merged["avg_gap_last3"] = (
    trans_merged.groupby("user_id")["time_diff"]
      .rolling(3)
      .mean()
      .shift(1)
      .reset_index(level=0, drop=True)
)


In [ ]:
# trans_merged = trans_merged.reset_index(drop=False)
# trans_merged['date'] = pd.to_datetime(trans_merged['date'])
# trans_merged = trans_merged.sort_values(["merchant_id", "date"])

#   merchant_fraud_rate_7d = (
#     trans_merged
#        .set_index("date")
#        .groupby("merchant_id")["is_fraud"]
#        .rolling("7D")
#        .mean()
#        .shift(1))

# merchant_fraud_rate_7d = merchant_fraud_rate_7d.reset_index(level=0, drop=True)
# trans_merged["merchant_fraud_rate_7d"] = merchant_fraud_rate_7d.values


In [ ]:
#trans_merged["mcc_fraud_rate_30d"] = (
#    trans_merged.groupby("mcc")["is_fraud"]
#      .rolling("30d")
#      .mean()
#      .shift(1)
#      .reset_index(level=0, drop=True)
#)

In [ ]:
user_hist = (
    trans_merged.groupby("user_id")["is_fraud"]
      .expanding()
      .mean()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["user_hist_fraud_rate"] = user_hist


In [ ]:
'''trans_merged = trans_merged.sort_values(["user_id", "date"])
last_fraud = (
    trans_merged[trans_merged["is_fraud"] == 1]
      .groupby("user_id")["date"]
      .shift(1)
)

trans_merged["days_since_last_fraud_user"] = (
    (trans_merged["date"] - last_fraud).dt.days
)'''

'trans_merged = trans_merged.sort_values(["user_id", "date"])\nlast_fraud = (\n    trans_merged[trans_merged["is_fraud"] == 1]\n      .groupby("user_id")["date"]\n      .shift(1)\n)\n\ntrans_merged["days_since_last_fraud_user"] = (\n    (trans_merged["date"] - last_fraud).dt.days\n)'

In [ ]:
trans_merged = trans_merged.sort_values(["user_id", "date"])

# Keep fraud dates, otherwise NaT
fraud_date = trans_merged["date"].where(trans_merged["is_fraud"].eq(1))

# For each user, carry the last seen fraud date forward
last_fraud_date = fraud_date.groupby(trans_merged["user_id"]).ffill()

# Shift so that on a fraud row, "last fraud" means the previous one (not itself)
last_fraud_date = last_fraud_date.groupby(trans_merged["user_id"]).shift(1)

trans_merged["days_since_last_fraud_user"] = (
    (trans_merged["date"] - last_fraud_date).dt.days
)

In [ ]:
trans_merged.loc[(trans_merged["user_id"].eq(100)) & (trans_merged["is_fraud"] == 1),
                 ["user_id","date","is_fraud","days_since_last_fraud_user"]].head(50)

,user_id,date,is_fraud,days_since_last_fraud_user
704313,100,2022-03-27 12:12:02,1,NaN
843659,100,2022-04-15 19:30:45,1,19.0
1036692,100,2022-05-10 08:58:17,1,24.0
1108692,100,2022-05-18 18:33:46,1,8.0
1120860,100,2022-05-20 04:50:06,1,1.0
1331131,100,2022-06-16 18:32:12,1,27.0
1443201,100,2022-07-01 23:15:07,1,15.0
1445881,100,2022-07-02 06:48:07,1,0.0
1477706,100,2022-07-06 00:39:03,1,3.0
1989232,100,2022-09-09 21:54:05,1,65.0


In [ ]:
col = "days_since_last_fraud_user"

# Add indicator first (recommended)
trans_merged[col + "_isnull"] = trans_merged[col].isna().astype(int)

# Fill NaNs with a large value
fill_value = trans_merged[col].max() + 1
trans_merged[col] = trans_merged[col].fillna(fill_value)

In [ ]:
trans_merged.loc[(trans_merged["user_id"].eq(100)) & (trans_merged["is_fraud"] == 1),
                 ["user_id","date","is_fraud","days_since_last_fraud_user"]].head(50)

,user_id,date,is_fraud,days_since_last_fraud_user
704313,100,2022-03-27 12:12:02,1,1090.0
843659,100,2022-04-15 19:30:45,1,19.0
1036692,100,2022-05-10 08:58:17,1,24.0
1108692,100,2022-05-18 18:33:46,1,8.0
1120860,100,2022-05-20 04:50:06,1,1.0
1331131,100,2022-06-16 18:32:12,1,27.0
1443201,100,2022-07-01 23:15:07,1,15.0
1445881,100,2022-07-02 06:48:07,1,0.0
1477706,100,2022-07-06 00:39:03,1,3.0
1989232,100,2022-09-09 21:54:05,1,65.0


In [ ]:
trans_merged.loc[trans_merged["is_fraud"] == 1]

,index,id,date,user_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,...,is_far_from_home,is_retired,z_amount_user,amount_user_median_ratio,hour_deviation_user,time_diff,avg_gap_last3,user_hist_fraud_rate,days_since_last_fraud_user,days_since_last_fraud_user_isnull
1410081,1410081,18424729,2022-06-26 00:56:20,0,1271,270.72,online transaction,83480,online,ONLINE,...,0,0,-0.590042,0.310231,1.0,1266.0,11666.666667,0.000000,1090.0,1
2665502,2665502,16879861,2022-12-04 23:51:10,0,4639,1101.24,online transaction,89705,online,ONLINE,...,0,0,-0.045567,1.249592,22.0,21659.0,15003.333333,0.000384,161.0,0
3567219,3567220,16879132,2023-03-26 08:52:40,0,4639,2329.20,online transaction,27092,online,ONLINE,...,0,0,0.748780,2.652183,6.0,13444.0,2840.333333,0.000577,111.0,0
4566411,4566411,16878036,2023-08-03 04:11:49,0,4639,92.34,chip transaction,85820,durban,kwazulu-natal,...,1,0,-0.696106,0.106786,10.0,2509.0,4558.333333,0.000675,129.0,0
5092143,5092143,16879843,2023-10-08 19:12:25,0,4639,1567.44,online transaction,23446,online,ONLINE,...,0,0,0.294589,1.808703,5.0,3798.0,7354.333333,0.000803,66.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1958735,1958735,16888692,2022-09-06 07:50:04,1998,2160,702.00,chip transaction,55076,polokwane,limpopo,...,1,0,0.205346,3.931452,3.0,83757.0,26389.000000,0.000000,1090.0,1
3337260,3337260,16888261,2023-02-27 10:48:41,1998,2160,284.04,online transaction,65881,online,ONLINE,...,0,0,-0.323625,1.552386,8.0,38310.0,23115.000000,0.001016,174.0,0
4219116,4219116,16887883,2023-06-18 14:19:21,1998,2160,5354.10,online transaction,90865,online,ONLINE,...,0,0,6.129810,29.479683,12.0,55089.0,5344.666667,0.001627,111.0,0
7027111,7027111,16878413,2024-06-11 01:02:58,1998,2160,2002.68,swipe transaction,15706,pretoria,gauteng,...,0,0,1.816616,11.059642,0.0,6264.0,41202.333333,0.001474,358.0,0


In [ ]:
trans_merged[["user_id", "date","is_fraud","days_since_last_fraud_user"]].sample(50)

,user_id,date,is_fraud,days_since_last_fraud_user
3995040,845,2023-05-20 09:22:00,0,27.0
6088655,1303,2024-02-13 10:45:41,0,326.0
5885642,1928,2024-01-17 14:01:30,0,12.0
5400524,366,2023-11-16 20:43:38,0,65.0
4509224,1523,2023-07-24 11:31:56,0,50.0
146699,545,2022-01-18 04:35:25,0,1090.0
7764574,1435,2024-09-13 10:21:37,0,543.0
114270,370,2022-01-14 09:26:15,0,1090.0
4098963,1145,2023-06-04 12:14:39,0,424.0
8134035,810,2024-10-28 16:01:00,0,24.0


In [ ]:
one_user = trans_merged["user_id"].iloc[0]
trans_merged[trans_merged["user_id"] == one_user][["date","is_fraud","days_since_last_fraud_user"]].head(50)

,date,is_fraud,days_since_last_fraud_user
293,2022-01-01 00:46:02,0,1090.0
1803,2022-01-01 05:02:57,0,1090.0
2113,2022-01-01 06:04:41,0,1090.0
2747,2022-01-01 07:49:43,0,1090.0
4939,2022-01-01 14:07:18,0,1090.0
5156,2022-01-01 14:46:33,0,1090.0
6362,2022-01-01 18:12:15,0,1090.0
6982,2022-01-01 20:00:49,0,1090.0
7901,2022-01-01 22:34:05,0,1090.0
10216,2022-01-02 05:03:52,0,1090.0


In [ ]:
trans_merged["days_since_last_fraud_user"].describe()

count    8.615533e+06
mean     2.891340e+02
std      3.935572e+02
min      0.000000e+00
25%      3.200000e+01
50%      9.300000e+01
75%      3.140000e+02
max      1.090000e+03
Name: days_since_last_fraud_user, dtype: float64

In [ ]:
'''df = trans_merged.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["user_id", "date"])

cols_to_show = ["user_id", "date", "id", "amount", "is_fraud", "days_since_last_fraud_user"]

for user_id, grp in df[cols_to_show].groupby("user_id"):
    print(f"\n========== User: {user_id} ==========")
    print(grp.sort_values("date").to_string(index=False))'''


'df = trans_merged.copy()\ndf["date"] = pd.to_datetime(df["date"])\ndf = df.sort_values(["user_id", "date"])\n\ncols_to_show = ["user_id", "date", "id", "amount", "is_fraud", "days_since_last_fraud_user"]\n\nfor user_id, grp in df[cols_to_show].groupby("user_id"):\n    print(f"\n========== User: {user_id} ==========")\n    print(grp.sort_values("date").to_string(index=False))'

In [ ]:
trans_merged["hour_sin"] = np.sin(2 * np.pi * trans_merged["trans_hour"] / 24)
trans_merged["hour_cos"] = np.cos(2 * np.pi * trans_merged["trans_hour"] / 24)

In [ ]:
trans_merged.isna().sum()

index                                0
id                                   0
date                                 0
user_id                              0
card_id                              0
                                    ..
user_hist_fraud_rate                 1
days_since_last_fraud_user           0
days_since_last_fraud_user_isnull    0
hour_sin                             0
hour_cos                             0
Length: 84, dtype: int64

In [ ]:
features_no_history = [
    'z_amount_user', 'amount_user_median_ratio',
    'hour_deviation_user', 'time_diff',
    'avg_gap_last3', 'user_hist_fraud_rate']

trans_merged[features_no_history] = trans_merged[features_no_history].fillna(0)
trans_merged.isna().sum().sort_values(ascending=False).head(70)

merchant_amt_sum_7d        1
user_txn_count_7d          1
user_amt_sum_24h           1
user_txn_count_24h         1
city_amt_sum_7d            1
                          ..
user_city_seen_before      0
is_first_time_user_city    0
user_hour_seen_before      0
is_first_time_user_hour    0
user_city                  0
Length: 70, dtype: int64

# Behavioral features

In [ ]:
'''trans_merged['is_unusual_brand_for_user'] = (
    trans_merged.groupby(['user_id', 'card_brand'])
      .cumcount()
      .eq(0)
      .astype(int)
)
user_txn_counts = trans_merged.groupby('user_id').cumcount()

trans_merged.loc[user_txn_counts < 3, 'is_unusual_brand_for_user'] = 0'''

In [ ]:
# Ensure datetime
'''trans_merged["date"] = pd.to_datetime(trans_merged["date"], errors="coerce")

def clean_error(e):
    try:
        if pd.isna(e):
            return "noerrors"
        e = str(e).lower().strip()
        e = e.replace(" ", "").replace("__", "_")
        return e
    except:
        return "noerrors"

# Apply row-by-row (safe, avoids memory blowup)
trans_merged["errors_clean"] = trans_merged["errors"].apply(clean_error)

# Convert to list safely
trans_merged["error_list"] = trans_merged["errors_clean"].apply(lambda x: x.split(","))

# Flag errors
trans_merged["has_error"] = (trans_merged["errors_clean"] != "noerrors").astype(int)'''

'trans_merged["date"] = pd.to_datetime(trans_merged["date"], errors="coerce")\n\ndef clean_error(e):\n    try:\n        if pd.isna(e):\n            return "noerrors"\n        e = str(e).lower().strip()\n        e = e.replace(" ", "").replace("__", "_")\n        return e\n    except:\n        return "noerrors"\n\n# Apply row-by-row (safe, avoids memory blowup)\ntrans_merged["errors_clean"] = trans_merged["errors"].apply(clean_error)\n\n# Convert to list safely\ntrans_merged["error_list"] = trans_merged["errors_clean"].apply(lambda x: x.split(","))\n\n# Flag errors\ntrans_merged["has_error"] = (trans_merged["errors_clean"] != "noerrors").astype(int)'

In [ ]:
features_no_history = [
    'is_high_amount', 'is_zscore_outlier', 'monthly_income',
    'amt_to_income_ratio', 'user_txn_count_24h', 'user_amt_sum_24h',
    'user_txn_count_7d', 'user_amt_sum_7d', 'merchant_txn_count_24h',
    'merchant_amt_sum_24h', 'merchant_txn_count_7d', 'merchant_amt_sum_7d',
    'city_txn_count_24h', 'city_amt_sum_24h', 'city_txn_count_7d',
    'city_amt_sum_7d', 'user_mcc_seen_before',
    'is_first_time_user_mcc', 'user_merchant_seen_before',
    'is_first_time_user_merchant', 'user_city_seen_before',
    'is_first_time_user_city', 'user_hour_seen_before',
    'is_first_time_user_hour'
]

trans_merged[features_no_history] = trans_merged[features_no_history].fillna(0)
trans_merged.isna().sum().sort_values(ascending=False).head(40)

index                    0
id                       0
date                     0
user_id                  0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state           0
zip                      0
mcc                      0
errors                   0
is_fraud                 0
description              0
card_brand               0
card_type                0
card_number              0
expires                  0
cvv                      0
has_chip                 0
num_cards_issued         0
credit_limit             0
acct_open_date           0
year_pin_last_changed    0
card_on_dark_web         0
current_age              0
retirement_age           0
birth_year               0
birth_month              0
gender                   0
address                  0
per_capita_income        0
yearly_income            0
total_debt               0
credit_score             0
num_credit_cards         0
i

In [ ]:
cols_to_drop = [
    'card_number', 'address', 'zip', 'description', 'merchant_state',
    'cvv', 'expires', 'birth_year', 'birth_month', 'retirement_age', 
    'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'yearly_income', 'total_debt', 'credit_score', 'user_city'
]

trans_merged = trans_merged.drop(columns=cols_to_drop)

In [ ]:
'''windows = {
    "1m": "1min",
    "5m": "5min",
    "1h": "1h",
    "24h": "24h"
}

for label, win in windows.items():
    s = (
        trans_merged
        .groupby("user_id")
        .rolling(win, on="date")["has_error"]
        .sum()
        .shift(1)                                # prevent leakage
        .reset_index(level=0, drop=True)         # drop group key; keep positional order
    )

trans_merged[f"errors_last_{label}"] = s.to_numpy()'''

'windows = {\n    "1m": "1min",\n    "5m": "5min",\n    "1h": "1h",\n    "24h": "24h"\n}\n\nfor label, win in windows.items():\n    s = (\n        trans_merged\n        .groupby("user_id")\n        .rolling(win, on="date")["has_error"]\n        .sum()\n        .shift(1)                                # prevent leakage\n        .reset_index(level=0, drop=True)         # drop group key; keep positional order\n    )\n\ntrans_merged[f"errors_last_{label}"] = s.to_numpy()'

In [ ]:
'''error_keywords = {
    "insufficient_balance": "insufficientbalance",
    "bad_pin": "badpin",
    "technical_glitch": "technicalglitch",
    "bad_card_number": "badcardnumber",
    "bad_expiration": "badexpiration",
    "bad_cvv": "badcvv",
    "bad_zipcode": "badzipcode"
}

for feature, keyword in error_keywords.items():
    trans_merged[f"{feature}_flag"] = trans_merged["errors"].fillna("").str.contains(keyword, case=False).astype(int)'''


'error_keywords = {\n    "insufficient_balance": "insufficientbalance",\n    "bad_pin": "badpin",\n    "technical_glitch": "technicalglitch",\n    "bad_card_number": "badcardnumber",\n    "bad_expiration": "badexpiration",\n    "bad_cvv": "badcvv",\n    "bad_zipcode": "badzipcode"\n}\n\nfor feature, keyword in error_keywords.items():\n    trans_merged[f"{feature}_flag"] = trans_merged["errors"].fillna("").str.contains(keyword, case=False).astype(int)'

In [ ]:
'''user_seen_errors = {}

def is_new_error(user, errors):
    if user not in user_seen_errors:
        user_seen_errors[user] = set()

    flat_err = set(errors) if errors != ["NoErrors"] else set()

    is_new = int(len(flat_err - user_seen_errors[user]) > 0)

    # update history
    user_seen_errors[user].update(flat_err)
    return is_new

trans_merged["is_new_error_code"] = trans_merged.apply(
    lambda row: is_new_error(row["user_id"], row["error_list"]),
    axis=1)'''

'user_seen_errors = {}\n\ndef is_new_error(user, errors):\n    if user not in user_seen_errors:\n        user_seen_errors[user] = set()\n\n    flat_err = set(errors) if errors != ["NoErrors"] else set()\n\n    is_new = int(len(flat_err - user_seen_errors[user]) > 0)\n\n    # update history\n    user_seen_errors[user].update(flat_err)\n    return is_new\n\ntrans_merged["is_new_error_code"] = trans_merged.apply(\n    lambda row: is_new_error(row["user_id"], row["error_list"]),\n    axis=1)'

In [ ]:
'''trans_merged = trans_merged.sort_values(["user_id", "date"])

trans_merged["is_success"] = (trans_merged["errors"] == "No Errors").astype(int)
trans_merged["seg"] = trans_merged.groupby("user_id")["is_success"].cumsum()

trans_merged["errors_before_success"] = (
    trans_merged
    .loc[trans_merged["has_error"] == 1]
    .groupby(["user_id", "seg"])
    .cumcount()
)

trans_merged["errors_before_success"] = (
    trans_merged["errors_before_success"].fillna(0).astype(int)'''


'trans_merged = trans_merged.sort_values(["user_id", "date"])\n\ntrans_merged["is_success"] = (trans_merged["errors"] == "No Errors").astype(int)\ntrans_merged["seg"] = trans_merged.groupby("user_id")["is_success"].cumsum()\n\ntrans_merged["errors_before_success"] = (\n    trans_merged\n    .loc[trans_merged["has_error"] == 1]\n    .groupby(["user_id", "seg"])\n    .cumcount()\n)\n\ntrans_merged["errors_before_success"] = (\n    trans_merged["errors_before_success"].fillna(0).astype(int)'

# Other Features

In [52]:
trans_merged.columns

Index(['index', 'id', 'date', 'user_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'mcc', 'errors', 'is_fraud',
       'card_brand', 'card_type', 'has_chip', 'num_cards_issued',
       'credit_limit', 'current_age', 'gender', 'per_capita_income',
       'num_credit_cards', 'is_negative_amount', 'abs_amount', 'day_of_week',
       'trans_hour', 'trans_weekend', 'trans_day', 'trans_month', 'trans_year',
       'is_high_amount', 'is_zscore_outlier', 'monthly_income',
       'amt_to_income_ratio', 'user_txn_count_24h', 'user_amt_sum_24h',
       'user_txn_count_7d', 'user_amt_sum_7d', 'merchant_txn_count_24h',
       'merchant_amt_sum_24h', 'merchant_txn_count_7d', 'merchant_amt_sum_7d',
       'city_txn_count_24h', 'city_amt_sum_24h', 'city_txn_count_7d',
       'city_amt_sum_7d', 'user_mcc_seen_before', 'is_first_time_user_mcc',
       'user_merchant_seen_before', 'is_first_time_user_merchant',
       'user_city_seen_before', 'is_first_time_user_city',
     

In [69]:
trans_merged["trans_time"] = trans_merged["date"].dt.time

In [71]:
# Required columns
# user_id, merchant_id, city, mcc
# trans_time (datetime64)
# amount
# is_fraud (0/1)
# latitude, longitude (optional but used if available)

trans_merged = trans_merged.sort_values(["user_id", "trans_time"]).reset_index(drop=True)

In [ ]:
# Previous transaction amount
trans_merged["prev_amount"] = trans_merged.groupby("user_id")["abs_amount"].shift(1)

trans_merged["amount_delta_last_txn"] = trans_merged["abs_amount"] - trans_merged["prev_amount"]
trans_merged["amount_pct_change"] = (trans_merged["abs_amount"] + 1) / (trans_merged["prev_amount"] + 1)

# Rolling mean (last 3)
trans_merged["mean_last3_amount"] = (
    trans_merged.groupby("user_id")["abs_amount"]
      .shift(1)
      .rolling(3)
      .mean()
)

trans_merged["amount_vs_last3_mean"] = (trans_merged["abs_amount"] + 1) / (trans_merged["mean_last3_amount"] + 1)

trans_merged["prev_amount"] = trans_merged["prev_amount"].fillna(trans_merged["abs_amount"])
trans_merged["mean_last3_amount"] = trans_merged["mean_last3_amount"].fillna(trans_merged["abs_amount"])


In [57]:
trans_merged.dtypes

index                             int64
id                                int64
date                     datetime64[ns]
user_id                           int64
card_id                           int64
                              ...      
amount_delta_last_txn           float64
amount_pct_change               float64
mean_last3_amount               float64
amount_vs_last3_mean            float64
prev_time                        object
Length: 75, dtype: object

In [75]:
trans_merged["trans_time"] = pd.to_datetime(trans_merged["trans_time"], errors="coerce").dt.time

In [ ]:
trans_merged["prev_time"] = (trans_merged.groupby("user_id")["trans_time"].shift(1))

trans_merged["time_diff_sec"] = (trans_merged["trans_time"] - trans_merged["prev_time"]).dt.total_seconds()

trans_merged["txn_rate_1h_vs_24h"] = (
    trans_merged["user_txn_count_24h"] /
    (trans_merged["user_txn_count_7d"] + 1))

trans_merged["txn_rate_24h_vs_7d"] = (
    trans_merged["user_txn_count_24h"] /
    (trans_merged["user_txn_count_7d"] + 1))


In [78]:
trans_merged["gap"] = trans_merged["time_diff_sec"]

trans_merged["mean_gap_last5"] = (
    trans_merged.groupby("user_id")["gap"]
        .shift(1)
        .rolling(5)
        .mean())

trans_merged["std_gap_last5"] = (
    trans_merged.groupby("user_id")["gap"]
        .shift(1)
        .rolling(5)
        .std())

trans_merged["gap_cv_last5"] = (
    trans_merged["std_gap_last5"] /
    (trans_merged["mean_gap_last5"] + 1))


In [ ]:
# Fraud count in last 7 days
trans_merged["user_fraud_count_7d"] = (
    trans_merged
    .set_index("date")      # full datetime → correct
    .groupby("user_id")["is_fraud"]
    .shift(1)               # prevents leakage
    .rolling("7D")          # rolling window by time, not rows
    .sum()
    .reset_index(level=0, drop=True)
)

# Days since last fraud
last_fraud_time = (
    trans_merged["date"]
    .where(trans_merged["is_fraud"] == 1)
)

last_fraud_time = last_fraud_time.groupby(trans_merged["user_id"]).ffill()

trans_merged["fraud_recency_score"] = np.exp(
    -trans_merged["days_since_last_fraud"] / 7
)


In [ ]:
def rolling_fraud_rate(group, window="7D"):
    fraud = group["is_fraud"].shift(1)
    total = fraud.rolling(window, on=group["trans_time"]).count()
    fraud_sum = fraud.rolling(window, on=group["trans_time"]).sum()
    return fraud_sum / (total + 1)

trans_merged["merchant_fraud_rate_7d"] = (
    trans_merged.groupby("merchant_id", group_keys=False)
        .apply(rolling_fraud_rate)
)

trans_merged["city_fraud_rate_24h"] = (
    trans_merged.groupby("city", group_keys=False)
        .apply(lambda x: rolling_fraud_rate(x, "1D"))
)

trans_merged["mcc_fraud_rate_7d"] = (
    trans_merged.groupby("mcc", group_keys=False)
        .apply(rolling_fraud_rate)
)

In [ ]:
trans_merged["first_time_and_high_amount"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["is_zscore_outlier"] > 2)
).astype(int)

trans_merged["first_time_and_far"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["is_far_from_home"] == 1)
).astype(int)

trans_merged["first_time_and_night"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["trans_hour"].between(0, 5))
).astype(int)

In [ ]:
from scipy.stats import entropy

def rolling_entropy(x):
    p = x.value_counts(normalize=True)
    return entropy(p)

trans_merged["user_city_entropy_30d"] = (
    trans_merged.groupby("user_id")["city_norm"]
        .shift(1)
        .rolling(30)
        .apply(rolling_entropy, raw=False)
)

In [ ]:
# Make sure trans_merged is sorted by user and time
trans_merged = trans_merged.sort_values(["user_id", "trans_time"])

# Previous city for the same user
trans_merged["prev_city"] = trans_merged.groupby("user_id")["city_norm"].shift(1)

# Detect city changes
trans_merged["city_changed"] = (trans_merged["city_norm"] != trans_merged["prev_city"]).astype(int)

# Time difference from previous transaction (in seconds)
trans_merged["time_diff_sec"] = (
    trans_merged["trans_time"] - trans_merged.groupby("user_id")["trans_time"].shift(1)
).dt.total_seconds()

# City change too fast (proxy for impossible travel)
trans_merged["city_change_fast"] = (
    (trans_merged["city_changed"] == 1) &
    (trans_merged["time_diff_sec"] < 3600)  # less than 1 hour
).astype(int)

# Night city changes (optional, high fraud signal)
trans_merged["is_night"] = trans_merged["trans_hour"].between(0, 5)
trans_merged["night_city_change"] = (
    (trans_merged["is_night"]) &
    (trans_merged["city_changed"] == 1)
).astype(int)

In [ ]:
trans_merged["amount_x_merchant_risk"] = (
    trans_merged["amount"] *
    trans_merged["merchant_fraud_rate_7d"]
)

trans_merged["night_x_velocity"] = (
    (trans_merged["trans_hour"].between(0, 5)) *
    trans_merged["txn_rate_1h_vs_24h"]
)


In [ ]:
trans_merged.replace([np.inf, -np.inf], np.nan, inplace=True)

num_cols = trans_merged.select_dtypes(include=[np.number]).columns
trans_merged[num_cols] = trans_merged[num_cols].fillna(0)


# More features 2

In [ ]:
trans_merged = trans_merged.sort_values(["user_id", "date"])

In [ ]:
trans_merged["prev_amount"] = trans_merged.groupby("user_id")["abs_amount"].shift(1)
trans_merged["amount_vs_last_txn"] = trans_merged["abs_amount"] / (trans_merged["prev_amount"] + 1e-6)

In [ ]:
trans_merged["user_amt_mean_last3"] = (
    trans_merged.groupby("user_id")["abs_amount"]
      .rolling(3, min_periods=1)
      .mean()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["amount_vs_avg_last3"] = trans_merged["abs_amount"] / (trans_merged["user_amt_mean_last3"] + 1e-6)

In [ ]:
trans_merged["prev_txn_time"] = trans_merged.groupby("user_id")["date"].shift(1)
trans_merged["time_diff_sec"] = (
    trans_merged["date"] - trans_merged["prev_txn_time"]
).dt.total_seconds()


In [ ]:
trans_merged["prev_gap"] = trans_merged.groupby("user_id")["time_diff_sec"].shift(1)
trans_merged["gap_ratio"] = trans_merged["time_diff_sec"] / (trans_merged["prev_gap"] + 1)

In [ ]:
trans_merged["txn_count_1h"] = (
    trans_merged.groupby("user_id")
      .rolling("1h", on="date")["abs_amount"]
      .count()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["txn_count_24h"] = (
    trans_merged.groupby("user_id")
      .rolling("24h", on="date")["abs_amount"]
      .count()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["txn_rate_change_1h_vs_24h"] = (
    trans_merged["txn_count_1h"] / (trans_merged["txn_count_24h"] + 1)
)

In [ ]:
# Ensure datetime
trans_merged["date"] = pd.to_datetime(trans_merged["date"])

# Sort REQUIRED for time-based rolling
trans_merged = trans_merged.sort_values(
    ["merchant_id", "date"]
).reset_index(drop=True)

# Shift to avoid leakage
trans_merged["is_fraud_shifted"] = (
    trans_merged.groupby("merchant_id")["is_fraud"].shift(1)
)

# 24-hour fraud rate
trans_merged["merchant_fraud_rate_24h"] = (
    trans_merged
        .groupby("merchant_id")
        .rolling("24h", on="date")["is_fraud_shifted"]
        .mean()
        .reset_index(level=0, drop=True)
)

# 7-day fraud rate
trans_merged["merchant_fraud_rate_7d"] = (
    trans_merged
        .groupby("merchant_id")
        .rolling("7d", on="date")["is_fraud_shifted"]
        .mean()
        .reset_index(level=0, drop=True)
)

ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
trans_merged.columns

Index(['index', 'id', 'date', 'user_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'mcc', 'errors', 'is_fraud',
       'card_brand', 'card_type', 'has_chip', 'num_cards_issued',
       'credit_limit', 'current_age', 'gender', 'per_capita_income',
       'num_credit_cards', 'is_negative_amount', 'abs_amount', 'day_of_week',
       'trans_hour', 'trans_weekend', 'trans_day', 'trans_month', 'trans_year',
       'is_high_amount', 'is_zscore_outlier', 'monthly_income',
       'amt_to_income_ratio', 'user_txn_count_24h', 'user_amt_sum_24h',
       'user_txn_count_7d', 'user_amt_sum_7d', 'merchant_txn_count_24h',
       'merchant_amt_sum_24h', 'merchant_txn_count_7d', 'merchant_amt_sum_7d',
       'city_txn_count_24h', 'city_amt_sum_24h', 'city_txn_count_7d',
       'city_amt_sum_7d', 'user_mcc_seen_before', 'is_first_time_user_mcc',
       'user_merchant_seen_before', 'is_first_time_user_merchant',
       'user_city_seen_before', 'is_first_time_user_city',
     

In [ ]:
trans_merged["fraud_decay_score_user"] = np.exp(-trans_merged["days_since_last_fraud_user"] / 7)
trans_merged["fraud_decay_score_user"] = trans_merged["fraud_decay_score_user"].fillna(0)

In [ ]:
trans_merged["far_home_x_new_merchant"] = (
    trans_merged["is_far_from_home"] * trans_merged["is_first_time_user_merchant"]
)

trans_merged["night_txn_x_amount"] = (
    ((trans_merged["trans_hour"] < 6) | (trans_merged["trans_hour"] > 22)).astype(int)
    * trans_merged["z_amount_user"]
)

In [ ]:
trans_merged["date"] = pd.to_datetime(trans_merged["date"])

trans_merged = (
    trans_merged
        .sort_values(["user_id", "date"])
        .reset_index(drop=True)
)

trans_merged["is_night_txn"] = (
    (trans_merged["trans_hour"] < 6) | (trans_merged["trans_hour"] > 22)
).astype(int)

trans_merged["night_txn_ratio_7d"] = (
    trans_merged.groupby("user_id")
      .rolling("7d", on="date")["is_night_txn"]
      .mean()
      .shift(1)
      .reset_index(level=0, drop=True)
)

MemoryError: Unable to allocate 2.95 GiB for an array with shape (46, 8615533) and data type float64

In [ ]:
feature_cols = [col for col in trans_merged.columns if col not in [
    "is_fraud",
    "date",
    "is_fraud_shifted"
]]

trans_merged[feature_cols] = trans_merged[feature_cols].replace([np.inf, -np.inf], np.nan)

MemoryError: Unable to allocate 2.76 GiB for an array with shape (43, 8615533) and data type float64

# Machine Learning

In [95]:
trans_merged.columns.tolist()

['index',
 'id',
 'date',
 'user_id',
 'card_id',
 'amount',
 'use_chip',
 'merchant_id',
 'merchant_city',
 'mcc',
 'errors',
 'is_fraud',
 'card_brand',
 'card_type',
 'has_chip',
 'num_cards_issued',
 'credit_limit',
 'current_age',
 'gender',
 'per_capita_income',
 'num_credit_cards',
 'is_negative_amount',
 'abs_amount',
 'day_of_week',
 'trans_hour',
 'trans_weekend',
 'trans_day',
 'trans_month',
 'trans_year',
 'is_high_amount',
 'is_zscore_outlier',
 'monthly_income',
 'amt_to_income_ratio',
 'user_txn_count_24h',
 'user_amt_sum_24h',
 'user_txn_count_7d',
 'user_amt_sum_7d',
 'merchant_txn_count_24h',
 'merchant_amt_sum_24h',
 'merchant_txn_count_7d',
 'merchant_amt_sum_7d',
 'city_txn_count_24h',
 'city_amt_sum_24h',
 'city_txn_count_7d',
 'city_amt_sum_7d',
 'user_mcc_seen_before',
 'is_first_time_user_mcc',
 'user_merchant_seen_before',
 'is_first_time_user_merchant',
 'user_city_seen_before',
 'is_first_time_user_city',
 'user_hour_seen_before',
 'is_first_time_user_hour'

In [99]:
print(len(trans_merged.columns))

93


In [ ]:
'''cols_to_remove = [
    'gender', 'errors', 'id', 'card_id', 'amount', 'merchant_id',
    'mcc', 'credit_limit', 'user_region', 'per_capita_income',
    'num_credit_cards', 'merchant_city', 'trans_day', 'trans_month','trans_year', 'tod_Afternoon', 'tod_Evening', 'tod_Morning', 'tod_Night',
    'is_retired', 'num_cards_issued', 'is_rapid_transaction_card', 'city_txn_count_7d', 'is_high_amount', 'is_first_time_user_hour',
    'is_rapid_transaction_user', 'error_list'
]

trans_final = trans_merged.drop(columns=cols_to_remove, errors='ignore')
trans_final.shape'''


In [100]:
cols_to_remove = [
    # -------------------------
    # Identifiers / High-cardinality (never model)
    # -------------------------
    'id',
    'card_id',
    'merchant_id',
    'mcc',
    'merchant_city',
    'user_region',

    # -------------------------
    # Raw amounts & superseded versions
    # -------------------------
    'amount',                    # abs_amount already used
    'credit_limit',
    'per_capita_income',

    # -------------------------
    # Demographics (low lift / regulatory risk)
    # -------------------------
    'gender',
    'is_retired',
    'num_cards_issued',
    'num_credit_cards',

    # -------------------------
    # Raw time components (replaced by sin/cos & ratios)
    # -------------------------
    'trans_day',
    'trans_month',
    'trans_year',
    'day_of_week',
    'trans_weekend',

    # -------------------------
    # Weak / noisy velocity features (keep ratios only)
    # -------------------------
    'time_diff',
    'time_diff_sec',
    'gap',
    'avg_gap_last3',
    'mean_gap_last5',
    'std_gap_last5',
    'gap_cv_last5',
    'user_txn_count_24h',
    'user_txn_count_7d',
    'city_txn_count_7d',

    # -------------------------
    # Weak amount-change proxies (dominated by ratios)
    # -------------------------
    'amount_delta_last_txn',
    'amount_pct_change',
    'mean_last3_amount',
    'is_high_amount',

    # -------------------------
    # Overlapping novelty encodings
    # -------------------------
    'is_first_time_user_hour',

    # -------------------------
    # One-hot time-of-day (replaced by night features)
    # -------------------------
    'tod_Afternoon',
    'tod_Evening',
    'tod_Morning',
    'tod_Night',

    # -------------------------
    # Error / rule-based features (usually brittle)
    # -------------------------
    'errors',
    'error_list',

    # -------------------------
    # Heuristic / unstable engineered flags
    # -------------------------
    'is_rapid_transaction_card',
    'is_rapid_transaction_user',

    # -------------------------
    # Leakage-prone summary
    # -------------------------
    'user_hist_fraud_rate'
]


In [104]:
trans_final = trans_merged.drop(columns=cols_to_remove, errors='ignore')
trans_final.shape

MemoryError: Unable to allocate 1.03 GiB for an array with shape (16, 8615533) and data type int64

In [ ]:
df.to_csv("output.csv", index=False)